In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("p3.ipynb")

# P3 — Housing Price Prediction and Assessment Fairness

Using Cook County housing data, you will build and compare prediction models, analyze assessment errors, and discuss their consequences for homeowners. Adapted from UC Berkeley’s Data 100, this project extends HW12 through feature engineering, preprocessing, validation, and fairness analysis.

Before beginning, watch the [Human Contexts and Ethics video](https://youtu.be/PvxMAukFvJE?feature=shared&t=80) and review the [slides on Canvas](https://mtu.instructure.com/courses/1619791/files/folder/projects?preview=137028488).

**Instructions:** Work in order, use the specified variable names, retain full precision, and preserve rows unless instructed otherwise. Use natural logarithms. Fit preprocessing and models on training data only; refit on all labeled data when instructed. Passing public tests does not guarantee full hidden-test credit. The Canvas deadline applies.

**Required files:** Keep `utility_func.py` beside the notebook. Provide `cook_county_train_val.csv` and `cook_county_contest_test.csv` in the same folder, either individually or inside `cook_county_data.zip`.

---
<a id="outline"></a>
## Project outline

### [Part I — Human context and ethics](#part1)

- [Question 1 — Stakeholder interests](#q1)
- [Question 2 — Assessment fairness](#q2)
- [Question 3 — Unequal burdens](#q3)

### [Part II — Data preparation and training diagnostics](#part2)

- [Question 4 — Split and baseline](#q4)
- [Question 5 — Predictor audit and interaction](#q5)

### [Part III — Explicit housing models and residual analysis](#part3)

- [Question 6 — Fit and compare models](#q6)
- [Question 7 — Validation residual plot](#q7)
- [Question 8 — Assessment-error patterns](#q8)

### [Part IV — Reusable pipelines and contest predictions](#part4)

- [Question 9 — Density and age features](#q9)
- [Question 10 — Compare feature sets](#q10)
- [Question 11 — Refit and export](#q11)

### [Part V — Assessment audits and homeowner consequences](#part5)

- [Question 12 — Price-group and interval audit](#q12)
- [Question 13 — Accuracy and fairness](#q13)

[Submission instructions](#submission)


---
### Import Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import linear_model as lm
from utility_func import *

plt.rcParams.update({'figure.figsize': (8,5), 'font.size': 11})
training_val_data = load_course_csv('cook_county_train_val.csv')
assert training_val_data.index.is_unique
assert {'Sale Price','Building Square Feet','Lot Size','Description','Age','Town Code','Garage 1 Size'}.issubset(training_val_data.columns)
print(f'Labeled sample: {len(training_val_data):,} rows')

---
<a id="part1"></a>
## Part I — Human context and ethics

- [Question 1 — Stakeholder interests](#q1)
- [Question 2 — Assessment fairness](#q2)
- [Question 3 — Unequal burdens](#q3)

[↑ Return to outline](#outline)

<!-- BEGIN QUESTION -->

---
<a id="q1"></a>
### Question 1 — Stakeholder interests
A house is sold, refinanced, and assessed for property taxes at different times. Identify **three parties** who may use an estimated value in these situations. For each party, explain whether a higher or lower valuation could be beneficial and why. State one case in which two parties' interests conflict.

**Rubric (4 points):** Three relevant stakeholders (1); valuation preferences with reasons (2); one conflict of interests (1).

_Type your answer here, replacing this text._

<!-- END QUESTION -->

[↑ Return to outline](#outline)


<!-- BEGIN QUESTION -->

---
<a id="q2"></a>
### Question 2 — Assessment fairness
Consider the following assessment outcomes. Choose **two** and explain which raises the greater fairness concern, including who could be affected. There is no single required ranking; justify your reasoning using the assigned video.

A. Similar homes receive different assessments because one owner's property record is outdated.
B. Every home is assessed 10% above its likely sale price.
C. Lower-priced homes are systematically overvalued while higher-priced homes are systematically undervalued.
D. An inaccurate assessment can be corrected only through an appeal process that some owners find difficult to access.

Distinguish an isolated error from a systematic pattern, and do not treat a predicted value as the final tax bill.

**Rubric (4 points):** Compares two scenarios and justifies the greater fairness concern (2); identifies affected people (1); distinguishes isolated errors from systematic or procedural inequity (1).

_Type your answer here, replacing this text._

<!-- END QUESTION -->

[↑ Return to outline](#outline)


---
## The CCAO Dataset

You'll work with the dataset from the Cook County Assessor's Office (CCAO) in Illinois. This government institution determines property taxes across most of Chicago's metropolitan areas and nearby suburbs. In the United States, all property owners must pay property taxes, which are then used to fund public services, including education, road maintenance, and sanitation. These property tax assessments are based on property values estimated using statistical models considering multiple factors, such as real estate value and construction cost.

However, this system is not without flaws. In late 2017, a lawsuit was filed against the office of Cook County Assessor Joseph Berrios for producing "[racially discriminatory assessments and taxes](https://www.chicagotribune.com/politics/ct-cook-county-board-assessor-berrios-met-20170718-story.html)." The lawsuit included claims that the assessor's office undervalued high-priced homes and overvalued low-priced homes, creating a visible divide along racial lines. Wealthy homeowners, who were typically white, [paid less in property taxes](https://fix8media-chicago.squarespace.com/bpnc-v-berrios-resource-page), whereas [working-class, non-white homeowners paid more](https://www.chicagotribune.com/news/breaking/ct-cook-county-assessor-berrios-sued-met-20171214-story.html).

The Chicago Tribune's four-part series, "[The Tax Divide](https://www.chicagotribune.com/investigations/ct-tax-divide-investigation-20180425-storygallery.html)," delves into how this was uncovered. After "compiling and analyzing more than 100 million property tax records from the years 2003 through 2015, along with thousands of pages of documents, then vetting the findings with top experts in the field," they discovered that "residential assessments had been so far off the mark for so many years." You can read more about their investigation [here](https://apps.chicagotribune.com/news/watchdog/cook-county-property-tax-divide/assessments.html).

Make sure to watch [the video](https://youtu.be/PvxMAukFvJE?feature=shared&t=80) before answering the following questions!

<!-- BEGIN QUESTION -->

---
<a id="q3"></a>
### Question 3 — Unequal assessment burdens
Using the assigned [video](https://youtu.be/PvxMAukFvJE?feature=shared&t=80) and the CCAO background above, explain **one way assessment errors can become unequal burdens across homeowners**. Describe one contributing issue in the data or assessment/appeal process and one change that could address it. Explain why reducing an overall error metric alone might not resolve the issue.

**Rubric (4 points):** Unequal burdens linked to video/context (1); contributing issue (1); corrective change (1); limits of overall error reduction (1).

_Type your answer here, replacing this text._

<!-- END QUESTION -->

[↑ Return to outline](#outline)


---
<a id="part2"></a>
## Part II — Data preparation and training diagnostics

- [Question 4 — Split and baseline](#q4)
- [Question 5 — Predictor audit and interaction](#q5)

[↑ Return to outline](#outline)

---
<a id="q4"></a>
### Question 4 — Partition the data and evaluate a baseline (14 points)

**A. Lock the partition.** Generate `shuffled_indices` using `np.random.RandomState` to create a seeded random generator (**use seed 2026**) and its `.permutation()` method. Set `cut = floor(0.75*n)`, take the first positions as `train_indices` and the rest as `validation_indices`, and create independent `train` and `validation` copies with `.iloc`. Preserve indexes/order and do not filter before splitting.

**B. Evaluate a constant baseline.** After your split cell, the provided cleaning cell retains finite sale prices above 499 and positive finite building areas, adds their natural logs, and supplies `base_train`, `base_valid`, `Y_train_m1`, `Y_valid_m1`. It does not drop missing Bedrooms/Age. Calculate `baseline_log_price` from the training mean log price, then calculate `baseline_train_rmse` and `baseline_valid_rmse` using that same constant. Do not compute a validation-specific constant.

Both model comparisons later reuse this split and retained population. The two answer cells are short; the intervening cleaning code is provided.


In [ ]:
shuffled_indices = ...
cut = ...
train_indices = ...
validation_indices = ...
train = ...
validation = ...

In [ ]:
# Provided population preparation; implementation is in utility_func.py.
base_train = prepare_labeled_partition(train)
base_valid = prepare_labeled_partition(validation)
Y_train_m1 = base_train['Log Sale Price'].copy()
Y_valid_m1 = base_valid['Log Sale Price'].copy()

In [ ]:
baseline_log_price = ...
baseline_train_rmse = ...
baseline_valid_rmse = ...

In [ ]:
grader.check("q4")

[↑ Return to outline](#outline)


In [ ]:
# Provided shared project views.
train_b, valid_b, labeled_b = prepare_project_views(training_val_data,base_train,base_valid)
test_data = load_course_csv('cook_county_contest_test.csv')
assert 'Sale Price' not in test_data and test_data.index.is_unique

In [ ]:
candidate_numeric = prepare_candidate_numeric(train_b)

In [ ]:
# Provided numeric extraction and Model 1 inputs.
raw_numeric_train = prepare_explicit_predictors(base_train)
raw_numeric_valid = prepare_explicit_predictors(base_valid)
assert raw_numeric_train.notna().any().all()
X_train_m1 = base_train[['Log Building Square Feet']].copy()
X_valid_m1 = base_valid[['Log Building Square Feet']].copy()
Y_train_m2, Y_valid_m2 = Y_train_m1.copy(), Y_valid_m1.copy()

---
<a id="q5"></a>
### Question 5 — Audit numeric predictors and build the interaction inputs (16 points)

**A. Training diagnostics.** The provided `candidate_numeric` matrix contains training-only `area`, `log_area`, `age`, and `bedrooms`. Create `training_feature_report`, indexed in that order, with `missing_fraction` over all training rows and pairwise-complete Pearson `correlation_with_log_price` against `np.log(train_b['Sale Price'])`. Set `selected_numeric_hint` to the greatest absolute correlation, breaking ties by feature order.

**B. Interaction-model inputs.** The provided `raw_numeric_train` and `raw_numeric_valid` have columns `log_area`, `bedrooms`, `age`. Compute `training_medians` on training rows only. Fill missing values in both matrices with these medians to create `X_train_m2` and `X_valid_m2`, then append `area_bedrooms = log_area * bedrooms` to each. Preserve rows/indexes and the stated column order. A short loop is allowed.

Do not calculate validation medians, scale predictors, drop observations, add an intercept column, or include the target. Model 1's one-column input and both models' response Series are provided. There is no location-summary task or extra function to write.


In [ ]:
training_feature_report = ...
selected_numeric_hint = ...

In [ ]:
training_medians = ...
X_train_m2 = ...
X_valid_m2 = ...
# Append the interaction to both matrices.
...

In [ ]:
grader.check("q5")

[↑ Return to outline](#outline)


---
<a id="part3"></a>
## Part III — Explicit housing models and residual analysis

- [Question 6 — Fit and compare models](#q6)
- [Question 7 — Validation residual plot](#q7)
- [Question 8 — Assessment-error patterns](#q8)

[↑ Return to outline](#outline)

In [ ]:
# Evaluation is imported from utility_func.py.
linear_model_m1 = lm.LinearRegression(fit_intercept=True)
linear_model_m2 = lm.LinearRegression(fit_intercept=True)

---
<a id="q6"></a>
### Question 6 — Fit and compare explicit housing models (12 points)

The two `lm.LinearRegression(fit_intercept=True)` objects and the evaluation helper are provided. Fit each model using its own training matrix and response. Store training fitted log prices as `Y_fitted_m1`, `Y_fitted_m2`, and validation predictions as `Y_predicted_m1`, `Y_predicted_m2`.

Call `summarize_log_predictions` with the dictionary `{'Model 1': Y_predicted_m1, 'Model 2': Y_predicted_m2}` and `Y_valid_m1`, storing the returned table as `model_scores`. Set `preferred_model` to the row with the lowest `valid_log_rmse`, selecting the first row in a tie. The helper computes log RMSE and dollar metrics; you do not need to reproduce its implementation.


In [ ]:
# Fit both provided models, then compute fitted values and validation predictions.
...
Y_fitted_m1 = ...
Y_fitted_m2 = ...
Y_predicted_m1 = ...
Y_predicted_m2 = ...
model_scores = ...
preferred_model = ...

In [ ]:
grader.check("q6")

[↑ Return to outline](#outline)


<!-- BEGIN QUESTION -->

---
<a id="q7"></a>
### Question 7 — Validation residual plot
Plot **Model 2's validation residuals against its predicted Log Sale Price** using `plt.scatter`. Residuals are `actual - predicted`. Label both axes, add a horizontal reference line at zero, and use small, transparent points to reduce overplotting. Use validation predictions, not training fitted values. This diagnostic can reveal changes in error spread and systematic prediction patterns.

**Rubric (8 points):** Uses Model 2 validation predictions on the x-axis (2); correctly computes and plots validation residuals as actual − predicted (2); labels both axes accurately (2); adds a horizontal zero line (1); uses small, transparent scatter points (1).

In [ ]:
plt.figure(figsize=(8, 5))
# Create the validation residual plot here.
...

<!-- END QUESTION -->

[↑ Return to outline](#outline)


---
<a id="q8"></a>
### Question 8 — Direction and frequency of assessment errors (12 points)

For Model 2, compute `median_log_price` as the unrounded median actual validation log price and `validation_residual = actual - predicted`. Split validation homes into a lower-price group (`actual <= median`) and a higher-price group (`actual > median`). Compute `lower_mean`, `higher_mean`, and the fractions with **strictly negative** residuals, `lower_over_fraction` and `higher_over_fraction`.

Set `assessment_pattern` to `'regressive'` if `lower_mean < 0 < higher_mean`, `'progressive'` if `higher_mean < 0 < lower_mean`, and `'mixed'` otherwise. A zero residual is not an overestimate. Do not round the threshold or drop observations. These diagnostics describe assessment-error patterns; they do not establish overall fairness or determine final tax bills.


In [ ]:
median_log_price = ...
validation_residual = ...
lower_mean = ...
higher_mean = ...
lower_over_fraction = ...
higher_over_fraction = ...
assessment_pattern = ...

In [ ]:
grader.check("q8")

[↑ Return to outline](#outline)


---
<a id="part4"></a>
## Part IV — Reusable pipelines and contest predictions

- [Question 9 — Density and age features](#q9)
- [Question 10 — Compare feature sets](#q10)
- [Question 11 — Refit and export](#q11)

[↑ Return to outline](#outline)

In [ ]:
# Provided deterministic inputs; parameters are fitted later by estimator.fit.
base_train_b = extract_base_features(train_b)
base_valid_b = extract_base_features(valid_b.drop(columns=['Sale Price']))
base_all_b = extract_base_features(labeled_b)
base_test_b = extract_base_features(test_data)
Y_train_final = np.log(train_b['Sale Price'])
Y_valid_final = np.log(valid_b['Sale Price'])
Y_all_final = np.log(labeled_b['Sale Price'])

---
<a id="q9"></a>
### Question 9 — Engineer density and nonlinear-age predictors (12 points)


The four supplied base matrices have identical six-column schemas. Create independent copies `X_train_final`, `X_valid_final`, `X_all_final`, `X_test_final`. For **each** append, in this order:

- `extra_feature = exp(log_area - log_lot)`: building-to-lot area ratio.
- `age_squared = (Age / 10) ** 2`: squared age in decades.

These features are different from the area-by-Bedrooms interaction used earlier. Preserve the six original columns, all row indexes, and their order. Keep missing entries missing; do not impute, scale, encode, fit parameters, or access target values here. A short loop is allowed. `X_all_final` is reserved for the final refit after model selection.


In [ ]:
X_train_final, X_valid_final = ..., ...
X_all_final, X_test_final = ..., ...
# Append the density and nonlinear-age features to all four matrices.
...

In [ ]:
grader.check("q9")

[↑ Return to outline](#outline)


In [ ]:
# Estimator construction is imported from utility_func.py.
assert X_train_final[['log_area','log_lot','Bedrooms','Age','extra_feature','age_squared']].notna().any().all()

---
<a id="q10"></a>
### Question 10 — Compare nested feature sets on held-out data (16 points)


Fit three provided estimator pipelines on **`X_train_final`, `Y_train_final` only**:

1. `baseline_model = make_valuation_model(baseline=True)`: log area only.
2. `structural_model = make_valuation_model(expanded=False)`: size, rooms, age and categorical predictors, excluding the two new features.
3. `rich_model = make_valuation_model()`: structural predictors plus density and squared age.

Store their validation log predictions in `baseline_valid_log`, `structural_valid_log`, `rich_valid_log`. Call `summarize_log_predictions` with the dictionary in order `baseline`, `structural`, `rich` and `Y_valid_final`, producing `validation_comparison`. Select `selected_candidate` using the lowest **dollar RMSE**, resolving ties by that order.

This comparison tests whether the engineered features improve held-out performance beyond the same structural model. Training fit alone cannot decide this. The utility pipelines fit preprocessing internally; do not learn it from validation data. No cross-validation, tuning, or new function definition is required.


In [ ]:
baseline_model = ...
structural_model = ...
rich_model = ...
baseline_valid_log = ...
structural_valid_log = ...
rich_valid_log = ...
validation_comparison = ...
selected_candidate = ...

In [ ]:
grader.check("q10")

[↑ Return to outline](#outline)


---
<a id="q11"></a>
### Question 11 — Refit the selected specification and export predictions (12 points)


Construct a **fresh** `final_model` matching Question 10's selected specification. Use `baseline=True` exactly when `selected_candidate == 'baseline'`, and `expanded=True` exactly when `selected_candidate == 'rich'`. Fit on `X_all_final`, `Y_all_final` only after model selection.

Predict `Y_test_pred` from `X_test_final` and create `contest_predictions` with a single `Log Sale Price` column and a default RangeIndex. Retain every contest row in original order, including missing predictors or unseen categories. Store finite natural-log predictions without exponentiation or clipping. The later audit must keep using the cached validation predictions from Question 10.


In [ ]:
final_model = ...
Y_test_pred = ...
contest_predictions = ...

In [ ]:
grader.check("q11")

[↑ Return to outline](#outline)


---
<a id="part5"></a>
## Part V — Assessment audits and homeowner consequences

- [Question 12 — Price-group and interval audit](#q12)
- [Question 13 — Accuracy and fairness](#q13)

[↑ Return to outline](#outline)

In [ ]:
# Preserve the cached held-out predictions for the unchanged manual audit.
Y_true = Y_valid_final.copy()
Y_pred = {'baseline':baseline_valid_log,'structural':structural_valid_log,'rich':rich_valid_log}[selected_candidate].copy()
preds_df, cheap_df, expensive_df = build_assessment_audit(Y_true,Y_pred)
median_Y_true = float(np.median(Y_true))
min_Y_true, max_Y_true = float(np.min(Y_true)), float(np.max(Y_true))

---
<a id="q12"></a>
### Question 12 — Audit held-out errors across price groups and intervals (12 points)

Use the **cached held-out predictions**, not final-model training predictions.

**A. Compare the two price groups.** For the provided `cheap_df` and `expensive_df`, calculate dollar RMSE as `rmse_cheap` and `rmse_expensive`, and fractions strictly underestimated as `prop_underest_cheap` and `prop_underest_expensive`. Exact predictions are not underestimates. Use every held-out row in each group.

**B. Audit smaller intervals.** Complete `rmse_interval(df, start, end)` and `prop_underest_interval(df, start, end)` using the input frame. Select `start <= True Log Sale Price < end`, returning dollar RMSE or the fraction strictly underestimated. Return `np.nan` for empty-interval RMSE and `-1` for an empty-interval fraction. Raise `ValueError` when `start >= end`; infinite endpoints are allowed. Do not modify `df` or hard-code model-dependent values.

Complete the two short answer cells. The interval boundaries must prevent adjacent groups from double-counting homes.


In [ ]:
rmse_cheap = ...
rmse_expensive = ...
prop_underest_cheap = ...
prop_underest_expensive = ...

In [ ]:
def rmse_interval(df, start, end):
    ...

def prop_underest_interval(df, start, end):
    ...

In [ ]:
grader.check("q12")

[↑ Return to outline](#outline)


<!-- BEGIN QUESTION -->

---
<a id="q13"></a>
### Question 13 — Accuracy, fairness, and safeguards
Suppose Model A has a slightly lower overall dollar RMSE, while Model B shows smaller differences in relative assessment errors between lower- and higher-priced homes. Which model would you recommend for further review, and what additional evidence would you require before using it for assessments?

Support your answer with the assigned video and one finding from your own assessment audit (Question 12). Discuss one limitation of sale-price groups as a fairness measure and one safeguard for homeowners. There is no single required model choice; we assess the reasoning, evidence, and engagement with the human context.

**Rubric (4 points):** Justified choice and further evidence (1); video and audit connection (1); fairness-measure limitation (1); homeowner safeguard (1).

_Type your answer here, replacing this text._

<!-- END QUESTION -->

[↑ Return to outline](#outline)


---
<a id="submission"></a>

## Congratulations on completing P3!

Complete all 13 questions, including written responses and plots. Restart the kernel, run all cells, and save your notebook. Use `grader.export()` to generate the submission ZIP, then upload it to P3 on Gradescope by the Canvas deadline.

[↑ Return to outline](#outline)

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)